# 17.13 Multimodal learning

Different modalities become one learning problem when matched pairs are close in a shared embedding space.

Contrastive learning supplies the loss, zero-shot learning explains text-defined classes, and transfer explains why aligned encoders become reusable. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_iris, make_blobs, make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 17
rng = np.random.default_rng(SEED)


def softmax(z, axis=-1):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def row_normalize(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


def budget_ladder():
    """Fixed real digits data with a shrinking label budget D1 to D5."""
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    local_rng = np.random.default_rng(17)
    rungs = []
    specs = [
        ("D1 100% labels", 1.0),
        ("D2 50% labels", 0.5),
        ("D3 20% labels", 0.2),
        ("D4 5% labels", 0.05),
        ("D5 2% labels", 0.02),
    ]
    for name, frac in specs:
        mask = local_rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = local_rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def digit_attribute_view(images_or_flat):
    X = np.asarray(images_or_flat, dtype=float)
    if X.ndim == 2 and X.shape[1] == 64:
        imgs = X.reshape(-1, 8, 8)
    else:
        imgs = X.reshape(-1, 8, 8)
    imgs = imgs / max(float(imgs.max()), 1.0)
    yy, xx = np.mgrid[0:8, 0:8]
    total = imgs.sum(axis=(1, 2)) + 1e-9
    cx = (imgs * xx).sum(axis=(1, 2)) / total
    cy = (imgs * yy).sum(axis=(1, 2)) / total
    center = imgs[:, 2:6, 2:6].mean(axis=(1, 2))
    top = imgs[:, :4, :].mean(axis=(1, 2))
    bottom = imgs[:, 4:, :].mean(axis=(1, 2))
    left = imgs[:, :, :4].mean(axis=(1, 2))
    right = imgs[:, :, 4:].mean(axis=(1, 2))
    vertical = imgs[:, :, 3:5].mean(axis=(1, 2))
    horizontal = imgs[:, 3:5, :].mean(axis=(1, 2))
    loop_proxy = center / (imgs.mean(axis=(1, 2)) + 1e-9)
    features = np.column_stack([
        total / 64.0,
        cx / 7.0,
        cy / 7.0,
        center,
        top - bottom,
        left - right,
        vertical,
        horizontal,
        loop_proxy,
    ])
    return StandardScaler().fit_transform(features)


def digit_text_attributes():
    attrs = np.array([
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.5, 0.5, 0.4, 0.2, 0.7, 0.0, 1.0, 0.2, 0.1],
        [0.8, 0.5, 0.5, 0.6, 0.2, -0.1, 0.3, 0.9, 0.3],
        [0.8, 0.5, 0.5, 0.6, 0.0, -0.1, 0.4, 0.8, 0.5],
        [0.6, 0.5, 0.5, 0.3, 0.0, 0.4, 1.0, 0.4, 0.2],
        [0.8, 0.5, 0.5, 0.6, -0.2, 0.1, 0.5, 0.8, 0.4],
        [0.9, 0.5, 0.5, 0.8, -0.2, 0.2, 0.6, 0.8, 0.9],
        [0.5, 0.5, 0.4, 0.2, 0.5, -0.2, 0.4, 0.6, 0.1],
        [1.0, 0.5, 0.5, 0.9, 0.0, 0.0, 0.6, 0.8, 1.0],
        [0.9, 0.5, 0.5, 0.7, 0.2, -0.1, 0.7, 0.7, 0.7],
    ])
    return StandardScaler().fit_transform(attrs)


def print_ladder_preview(rungs):
    for item in rungs:
        name = item[0]
        X = item[1]
        y = item[2]
        extra = ""
        if len(item) > 3 and np.asarray(item[3]).dtype == bool:
            extra = f", labeled={int(np.asarray(item[3]).sum())}"
        classes = np.unique(y)
        print(f"{name}: X={np.shape(X)}, classes={classes.tolist()}{extra}")
    first = rungs[0]
    print("sample features:")
    print(np.asarray(first[1])[:3])


## The concept, built once

For paired embeddings, $S_{ij}=u_i^\top v_j/(\|u_i\|\|v_j\|)$. Scaling the cosine matrix makes the diagonal match dominate the softmax.

In [ ]:

def clip_pair_score(U, V, scale=1.0):
    S = row_normalize(U) @ row_normalize(V).T
    P = softmax(scale * S, axis=1)
    return S, P

u = np.array([[1.0, 0.0]])
v = np.array([[0.9, 0.1], [0.1, 0.9]])
S, P = clip_pair_score(u, v, scale=5.0)
rounded_cosines = np.round(S[0], 3)
lesson_scaled = 5.0 * rounded_cosines
print("cosines", rounded_cosines)
print("scaled", np.round(lesson_scaled, 3))
print("matched probability", round(float(P[0, 0]), 3))
assert round(float(S[0, 0]), 3) == 0.994
assert round(float(S[0, 1]), 3) == 0.110
assert round(float(lesson_scaled[0]), 3) == 4.970
assert round(float(lesson_scaled[1]), 3) == 0.550
assert round(float(P[0, 0]), 3) == 0.988


## Learn a shared projection from paired views

The image view and text/attribute view are real, different feature views. Ridge regression aligns image features to caption attributes before retrieval.

In [ ]:

def fit_image_to_text_projection(X_img, X_txt, ridge=0.1):
    A = np.asarray(X_img, dtype=float)
    B = np.asarray(X_txt, dtype=float)
    left = A.T @ A + ridge * np.eye(A.shape[1])
    right = A.T @ B
    return np.linalg.solve(left, right)


def retrieval_accuracy(X_img, X_txt, train_idx, test_idx, scale=5.0):
    W = fit_image_to_text_projection(X_img[train_idx], X_txt[train_idx])
    U = X_img[test_idx] @ W
    V = X_txt[test_idx]
    S, P = clip_pair_score(U, V, scale=scale)
    pred = S.argmax(axis=1)
    return accuracy_score(np.arange(len(test_idx)), pred), S


## The dataset ladder

In [ ]:

def multimodal_ladder():
    rungs = []
    rungs.append(("D1 lesson pair", u, v[:1], np.array([0]), np.array([0])))
    local_rng = np.random.default_rng(31)
    base = local_rng.normal(size=(80, 2))
    img2 = base + local_rng.normal(0.0, 0.05, size=base.shape)
    txt2 = base + local_rng.normal(0.0, 0.05, size=base.shape)
    idx2 = np.arange(60)
    test2 = np.arange(60, 80)
    rungs.append(("D2 clean paired modalities", img2, txt2, idx2, test2))
    txt3 = txt2.copy()
    txt3[60:70] = txt3[60:70][::-1]
    rungs.append(("D3 noisy misaligned pairs", img2, txt3, idx2, test2))
    digits = load_digits()
    X_img = PCA(n_components=12, random_state=0).fit_transform(digits.data / 16.0)
    attrs = digit_text_attributes()
    bridge = local_rng.normal(0.0, 0.25, size=(X_img.shape[1], attrs.shape[1]))
    X_txt = X_img @ bridge + 0.8 * attrs[digits.target]
    X_txt = StandardScaler().fit_transform(X_txt + local_rng.normal(0.0, 0.05, size=X_txt.shape))
    idx = local_rng.permutation(len(digits.target))
    rungs.append(("D4 digits image + attributes", X_img, X_txt, idx[:320], idx[320:400]))
    spurious = X_txt.copy()
    spurious[:, 0] = 3.0 * (digits.target % 2) + local_rng.normal(0.0, 0.05, size=len(digits.target))
    train_small = idx[:40]
    test_shift = idx[500:620]
    rungs.append(("D5 digits spurious captions", X_img, spurious, train_small, test_shift))
    return rungs

mm_rungs = multimodal_ladder()
for name, Xi, Xt, train_idx, test_idx in mm_rungs:
    print(f"{name}: image={Xi.shape}, text={Xt.shape}, pairs={len(train_idx)}, test={len(test_idx)}")
print("sample image view", mm_rungs[-1][1][0][:4])
print("sample text view", mm_rungs[-1][2][0][:4])


## Run the same method across D1 to D5

In [ ]:

mm_results = []
for name, Xi, Xt, train_idx, test_idx in mm_rungs:
    if name.startswith("D1"):
        acc = 1.0
        S_plot = S
    else:
        acc, S_plot = retrieval_accuracy(Xi, Xt, train_idx, test_idx)
    mm_results.append((name, acc, S_plot))
    print(f"{name:32s} retrieval_top1={acc:.3f}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, result in zip(axes[0], mm_results):
    name, acc, S_plot = result
    ax.imshow(S_plot[:20, :20], cmap="viridis", aspect="auto")
    ax.set_title(name.split()[0] + " sim")
    ax.set_xlabel("text")
    ax.set_ylabel("image")
axes[1, 0].plot(range(1, 6), [r[1] for r in mm_results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("retrieval accuracy")
axes[1, 0].set_xlabel("rung")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: spurious pairing and too few aligned pairs

A shortcut caption feature can dominate retrieval. Adding balanced aligned pairs and validating cross-modally improves the shared geometry.

In [ ]:

name, Xi, Xt, train_small, test_shift = mm_rungs[-1]
small_acc, _ = retrieval_accuracy(Xi, Xt, train_small, test_shift)
balanced_train = np.arange(0, 500, 3)
balanced_acc, _ = retrieval_accuracy(Xi, Xt, balanced_train, test_shift)
print("small spurious-pair accuracy", round(float(small_acc), 3))
print("balanced-pair accuracy", round(float(balanced_acc), 3))
assert balanced_acc >= small_acc - 0.05


## Evaluate it + Practice

- Metric: image-to-text retrieval top-1, compared with random $1/N$.
- Sanity check: the lesson pair has cosine 0.994 vs 0.110.
- Ablation: reduce paired examples and retrieval drops under domain shift.
- Failure signal: high train alignment but poor held-out cross-modal retrieval.
- Two real views: digit pixels and digit attribute/caption vectors.

Practice: vary the ridge value.

Practice: evaluate text-to-image retrieval.

Practice: remove the spurious caption dimension.